[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kaushikreddym/climdata/blob/main/docs/examples/datasets/cmip_w5e5_example.ipynb)

In [ ]:
# Uncomment to install climdata in Google Colab or other environments
# !pip install climdata

## Example 1: Basic Usage - Point Extraction

Extract climate projection data for a specific location.

In [ ]:
from climdata import ClimData
from climdata.datasets.CMIP_W5E5 import CMIPW5E5

model = 'gfdl-esm4'

for scen in ['ssp126', 'ssp370']:

    # Configure for Berlin, Germany
    overrides = [
        "dataset=cmip_w5e5",                  # Select CMIP-W5E5 dataset
        "lat=52.52",                          # Berlin latitude
        "lon=13.40",                          # Berlin longitude
        f"experiment_id={scen}",              # Climate scenario
        f"source_id={model}",                 # CMIP6 model
        "time_range.start_date=2015-01-01",   # Future period
        "time_range.end_date=2015-12-31",
        "variables=[tas,tasmin,tasmax,pr,rsds,hurs,sfcWind]",
        "data_dir=./data",                    # Local directory for downloaded files
    ]

    extractor = ClimData(overrides=overrides)

    # Initialize CMIP-W5E5 dataset
    cmip_w5e5 = CMIPW5E5(extractor.cfg)

    # Fetch data from ISIMIP repository
    cmip_w5e5.fetch()

## Example 2: End-to-End Workflow

Chain extraction, imputation, index calculation and NetCDF export in a single call.

In [ ]:
overrides = [
    "dataset=cmip_w5e5",
    "lat=52",
    "lon=13",
    "time_range.start_date=1989-01-01",  # Start date for data extraction
    "time_range.end_date=1989-12-31",    # End date for data extraction
    "variables=[tasmin]",
    "data_dir=./data",                   # Local directory to store raw/intermediate files
    "source_id=GFDL-ESM4",
    "index=tn10p",                       # Climate extreme index to calculate
    "impute=BRITS",                      # Deep-learning gap filling
]

# The workflow runs each action in order
seq = ["extract", "impute", "calc_index", "to_nc"]

extractor = ClimData(overrides=overrides)
result = extractor.run_workflow(actions=seq)

## Example 3: Explore Available Models and Scenarios

Discover what CMIP6 experiments and models are available.

In [ ]:
cmip_w5e5 = CMIPW5E5(extractor.cfg)

# Get available experiment IDs
experiments = cmip_w5e5.get_experiment_ids()
print("Available CMIP6 Experiments:")
for exp in experiments:
    print(f"  - {exp}")

In [ ]:
# Get available models for a specific experiment
models = cmip_w5e5.get_source_ids(experiment_id='ssp585')
print("\nAvailable Models for SSP5-8.5:")
for model in models:
    print(f"  - {model}")

## Example 4: Compare Multiple Scenarios

Compare low and high emission scenarios for the same location.

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
from climdata import ClimData
from climdata.datasets.CMIP_W5E5 import CMIPW5E5
# Function to load data for a scenario
def load_scenario(scenario, model='gfdl-esm4'):
    overrides = [
        "dataset=cmip_w5e5",
        "lat=52.52",
        "lon=13.40",
        f"experiment_id={scenario}",
        f"source_id={model}",
        "time_range.start_date=2050-01-01",
        "time_range.end_date=2050-12-31",
        "variables=[tasmin]",
        "data_dir=./data",
    ]
    extractor = ClimData(overrides=overrides)
    cmip = CMIPW5E5(extractor.cfg)
    cmip.fetch()
    cmip.load()
    cmip.extract(point=(extractor.cfg.lon, extractor.cfg.lat))
    return cmip.ds

# Load low and high emission scenarios
ds_low = load_scenario('ssp126')  # Low emissions
ds_high = load_scenario('ssp585') # High emissions

# Plot comparison
plt.figure(figsize=(12, 6))
ds_low['tasmin'].plot(label='SSP1-2.6 (Low emissions)', alpha=0.7)
ds_high['tasmin'].plot(label='SSP5-8.5 (High emissions)', alpha=0.7)
plt.title('Future Temperature Projections for Berlin')
plt.ylabel('Temperature (K)')
plt.xlabel('Year')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Example 5: Regional Extraction with Bounding Box

Extract data for a region instead of a single point.

In [ ]:
from climdata.datasets.CMIP_W5E5 import CMIPW5E5
from climdata import ClimData

# Configure for Central Europe region
overrides = [
    "dataset=cmip_w5e5",
    "experiment_id=ssp126",              # Low emissions scenario
    "source_id=gfdl-esm4",               # NOAA GFDL Earth System Model
    "time_range.start_date=2050-01-01",
    "time_range.end_date=2050-12-31",
    "variables=[tas,pr]",
    "data_dir=./data",
]

extractor = ClimData(overrides=overrides)
cmip_w5e5 = CMIPW5E5(extractor.cfg)

# Fetch and load
cmip_w5e5.fetch()
cmip_w5e5.load()

# Extract for Central Europe bounding box
box = {
    'lon_min': 5.0,   # Western boundary
    'lon_max': 15.0,  # Eastern boundary
    'lat_min': 47.0,  # Southern boundary
    'lat_max': 55.0   # Northern boundary
}
cmip_w5e5.extract(box=box)

# View spatial data
cmip_w5e5.ds

In [ ]:
# Plot a spatial map for a single day
import hvplot.xarray  # noqa: F401

cmip_w5e5.ds['tas'].isel(time=0).hvplot(coastline=True, geo=True)

## Example 6: Save Data to File

Export extracted data to NetCDF or CSV format.

In [ ]:
from climdata.datasets.CMIP_W5E5 import CMIPW5E5
from climdata import ClimData

# Configure and extract
overrides = [
    "dataset=cmip_w5e5",
    "lat=52.52",
    "lon=13.40",
    "experiment_id=ssp370",
    "source_id=mpi-esm1-2-hr",
    "time_range.start_date=2080-01-01",
    "time_range.end_date=2080-12-31",
    "variables=[tas,pr,rsds]",
    "data_dir=./data",
]

extractor = ClimData(overrides=overrides)
cmip_w5e5 = CMIPW5E5(extractor.cfg)
cmip_w5e5.fetch()
cmip_w5e5.load()
cmip_w5e5.extract(point=(extractor.cfg.lon, extractor.cfg.lat))

# Save as NetCDF
cmip_w5e5.save_netcdf('berlin_ssp370_2080.nc')

# Save as CSV
cmip_w5e5.save_csv('berlin_ssp370_2080.csv')

print("✅ Data saved successfully!")

## Example 7: Using the ClimData Workflow

Use the high-level `ClimData` workflow for end-to-end processing.

In [ ]:
from climdata import ClimData

# Configure with overrides
overrides = [
    "dataset=cmip_w5e5",
    "lat=12.89",
    "lon=24.25",
    "experiment_id=ssp585",
    "source_id=gfdl-esm4",
    "time_range.start_date=2050-01-01",
    "time_range.end_date=2050-12-31",
    "variables=[tasmin,tasmax,pr]",
    "data_dir=./data",
]

# Initialize extractor
extractor = ClimData(overrides=overrides)

# Extract data
ds = extractor.extract()

# Convert to DataFrame
df = extractor.to_dataframe()

# Save results
result = extractor.save(format='csv')

print(f"Data extracted and saved to: {result.filename}")
df.head()

## Example 8: Calculate Climate Indices

Compute extreme indices like frost days or heat waves from CMIP-W5E5 data.

In [ ]:
from climdata import ClimData

# Configure with climate index
overrides = [
    "dataset=cmip_w5e5",
    "lat=52.52",
    "lon=13.40",
    "experiment_id=ssp585",
    "source_id=gfdl-esm4",
    "time_range.start_date=2050-01-01",
    "time_range.end_date=2079-12-31",  # 30 years for climate indices
    "variables=[tasmin,tasmax,pr]",
    "index=tn10p",  # Cold nights (10th percentile of minimum temperature)
    "data_dir=./data",
]

extractor = ClimData(overrides=overrides)

# Extract base data
ds = extractor.extract()

# Calculate index
index_ds = extractor.calc_index()

# View results
print("Climate index calculated:")
index_ds

## Key Features

### Scenarios Available
- **historical**: Historical period (typically 1850-2014)
- **ssp126**: Low emissions (sustainability pathway)
- **ssp245**: Middle-of-the-road (moderate emissions)
- **ssp370**: Medium-high emissions
- **ssp585**: High emissions (fossil-fueled development)

### Models Available in ISIMIP3b
- **gfdl-esm4**: NOAA Geophysical Fluid Dynamics Laboratory
- **ipsl-cm6a-lr**: Institut Pierre-Simon Laplace
- **mpi-esm1-2-hr**: Max Planck Institute for Meteorology
- **mri-esm2-0**: Meteorological Research Institute
- **ukesm1-0-ll**: UK Earth System Modeling

### Variables
Standard CMIP6 variables are available:
- **tas**: Near-surface air temperature (K)
- **tasmin**: Daily minimum temperature (K)
- **tasmax**: Daily maximum temperature (K)
- **pr**: Precipitation (kg m⁻² s⁻¹)
- **rsds**: Surface downwelling shortwave radiation (W m⁻²)
- **hurs**: Near-surface relative humidity (%)
- **sfcWind**: Near-surface wind speed (m s⁻¹)
- **ps**: Surface air pressure (Pa)

## Notes

1. **Data Size**: CMIP6 files can be large. Start with short time periods for testing.
2. **ISIMIP Client**: Requires `isimip-client` package for data access.
3. **Resolution**: Data is provided at 0.5° (~55 km) spatial resolution.
4. **Bias Adjustment**: ISIMIP3b data is bias-adjusted to W5E5 observations.
5. **Attribution**: When using this data, cite both CMIP6 and ISIMIP appropriately.

## References

- ISIMIP: https://www.isimip.org/
- CMIP6: https://www.wcrp-climate.org/wgcm-cmip/wgcm-cmip6
- W5E5: https://doi.org/10.48364/ISIMIP.342217